# 09 · Integrated AHR v2 XL and full-model recovery gate

**Status: UNEXECUTED FIRST DRAFT.** This notebook contains working implementations to inspect and refine when run. No trained capability or benchmark result is asserted.

**Purpose:** Join the full encoder, world model, Queen, Worker, Scout and literal Lorentz workspace into a bounded interactive planner and validate optimizer continuation before any long training.

**Inputs:** Notebooks 00–08; installed PyTorch environment; passed notebook01 recovery gate; full XL configuration. No dataset or external game service is needed for the synthetic recovery check.

**Outputs:** AHRModel, action planner, validate_full_model_recovery and require_full_model_gate; actual parameter/profile report; recovery checkpoints and pass/fail report when the user runs this notebook.

**Expected runtime:** Expect several minutes for full allocation, three optimizer updates and checkpoint I/O; profile first. The 12-minute runtime budget is checked between steps. Slow hardware can overrun one atomic update or save. No long training is launched. These are planning estimates; measure on this instance.

**Checkpoint directory:** `checkpoints/<run_id>/` under the project root. Definitions in this notebook are included in every source manifest.

**Exact resume:** Run 00, then 01, then restart this notebook and Run All. Each recovery attempt has a unique run ID and reports its actual outcome. A failed test is rerun from its controlled fixture; it does not silently resume a game or a production run. Training notebooks resume their own latest valid checkpoints only after both gates pass.

The configuration remains the full XL design (`d_model=1024` by default). Any explicitly identified synthetic fixture verifies plumbing only and is not a smaller AHR training configuration.


## Load explicit notebook dependencies
Only tagged definitions are loaded; dependency execution cells are not run.

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([2, 3, 8], globals())

## Differentiable full model

The three roles share semantic context and a goal. The action policy combines Queen and lower-role logits so policy losses train the proposals actually used in planning. Geometry remains a separate 1+3 event representation. No compact model is substituted.

In [ ]:
class AHRModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = copy.deepcopy(cfg)
        r = cfg['reasoning']
        if int(r['max_depth']) < 1 or int(r['max_model_calls']) < 2:
            raise ValueError('Planning requires depth>=1 and at least encoder+Queen calls.')
        per_parent = int(r['worker_candidates']) + (int(r['scout_candidates']) if r['scout_enabled'] else 0)
        required_capacity = 1 + 2*int(r['max_depth'])*per_parent
        if int(r['memory_capacity']) < required_capacity:
            raise ValueError(f'memory_capacity must be >= {required_capacity} to retain every current search parent; reduce bounded candidates/depth or increase memory.')
        self.encoder = GridEncoder(cfg)
        self.world = LatentWorldModel(cfg)
        self.worker = WorkerEBM(cfg)
        self.scout = ScoutEBM(cfg)
        self.queen = QueenController(cfg)
        self.geometry = EventGeometry(cfg)

    def forward(self, grids, actions=None, xy=None):
        tokens, latent = self.encoder(grids)
        mask = torch.ones(latent.shape[0], int(self.cfg['model']['action_dim']), device=latent.device, dtype=torch.bool)
        # Training supplies real public masks in its loss; RESET remains a valid output.
        decision = self.queen(latent.unsqueeze(1), latent.unsqueeze(1), mask)
        worker = self.worker.direct(latent, decision['goal'])
        scout = self.scout.direct(latent, decision['goal'])
        worker_weight = decision['worker_weight']
        scout_weight = decision['scout_weight']
        if not self.cfg['reasoning']['scout_enabled']:
            worker_weight = torch.ones_like(worker_weight)
            scout_weight = torch.zeros_like(scout_weight)
        action_logits = decision['action_logits'] + worker_weight[:,None]*worker['action_logits'] + scout_weight[:,None]*scout['action_logits']
        click_logits = decision['click_logits'] + worker_weight[:,None,None]*worker['click_logits'] + scout_weight[:,None,None]*scout['click_logits']
        result = dict(decision)
        result.update({'action_logits': action_logits, 'click_logits': click_logits,
                       'worker_weight': worker_weight, 'scout_weight': scout_weight,
                       'latent': latent, 'worker_proposal': worker['latent'], 'scout_proposal': scout['latent'],
                       'worker_energy': self.worker.energy(latent, worker['latent'], decision['goal']),
                       'scout_energy': self.scout.energy(latent, scout['latent'], decision['goal']),
                       'geometry_regularizer': self.geometry.regularizer(latent)})
        if actions is not None:
            if xy is None:
                if bool((torch.as_tensor(actions) == 6).any()):
                    raise ValueError('ACTION6 world prediction requires explicit xy.')
                xy = torch.zeros(latent.shape[0], 2, device=latent.device, dtype=latent.dtype)
            predicted = self.world(latent, actions, xy)
            result['world'] = predicted
            result.update(predicted)
        return result

    def reset_episode(self):
        # Episode history is explicit: callers discard the returned episode_state on reset.
        return None

    def act(self, observation, available_actions, episode_state=None, budget=None):
        return plan_ahr_action(self, observation, available_actions, episode_state, budget)


def _policy_mask(available_actions, device):
    ids = [int(getattr(a, 'value', a)) for a in available_actions]
    if any(a < 0 or a > 7 for a in ids):
        raise ValueError('Unexpected public action ID.')
    if any(a != 0 for a in ids):
        ids = [a for a in ids if a != 0]
    mask = torch.zeros(1, 8, dtype=torch.bool, device=device)
    if ids:
        mask[0, ids] = True
    return validate_action_mask(mask, 1)


def _action_from_candidate(action_id, normalized_xy):
    action_id = int(action_id)
    if action_id == 6:
        pixels = normalized_xy_to_pixels(torch.as_tensor(normalized_xy)).reshape(2).tolist()
        return ActionSpec(action=6, x=int(pixels[0]), y=int(pixels[1]))
    return ActionSpec(action=action_id)

## Bounded Queen–Worker–Scout planning

The public observation is the only source of observed events. Planning branches roll forward through the learned world model and are always labeled imagined. Every root action must pass the actual public mask; future masks are held as explicit hypotheses. The finite search uses a two-branch beam, fixed maximum depth and a reserved neural-call budget. Construction requires memory capacity >= 1 + 2 × depth × sum(role candidate maxima), so pruning cannot remove a current search parent; incompatible configurations fail explicitly.

In [ ]:
def plan_ahr_action(model, observation, available_actions, episode_state=None, budget=None):
    started = time.monotonic()
    cfg, device = model.cfg, next(model.parameters()).device
    reason = cfg['reasoning']
    raw_observation = observation if isinstance(observation, dict) else {'grid': observation}
    observation_source = raw_observation.get('observation_source', 'public_environment_observation' if raw_observation.get('game_id') is not None else 'caller_provided_grid')
    grid = validate_grid_tensor(raw_observation['grid']).to(device)
    if grid.shape[0] != 1:
        raise ValueError('act accepts one public observation at a time.')
    mask = _policy_mask(available_actions, device)
    maximum = int(reason['max_model_calls'] if budget is None else min(int(budget), int(reason['max_model_calls'])))
    ledger = ReasoningBudget(maximum)
    state = copy.deepcopy(episode_state) if episode_state is not None else {}
    identity = raw_observation.get('episode_id')
    if identity is not None and state.get('episode_id') not in (None, identity):
        raise ValueError('Episode identity changed. Pass episode_state=None after a reset.')
    workspace = EventWorkspace(int(reason['memory_capacity']))
    if state.get('workspace'):
        workspace.load_state_dict(state['workspace'])
    was_training = model.training
    model.eval()
    trace = {'status': 'untrained_or_trained_status_is_callers_responsibility', 'candidates': [],
             'observation_source': observation_source, 'future_action_masks': 'current mask reused as unverified hypothesis'}
    try:
        with torch.no_grad():
            if not ledger.consume('encoder'):
                raise RuntimeError('Missing encoder budget.')
            _, latent = model.encoder(grid)
            parent_observed = workspace.get(workspace.last_observed_id)
            parent_coordinates = None if parent_observed is None else parent_observed['coordinates']
            coordinates = model.geometry.event(latent, parent_coordinates)[0]
            provenance = {'source': observation_source, 'game_id': raw_observation.get('game_id'),
                          'observation_hash': canonical_hash(grid[0].detach().cpu().tolist()),
                          'available_actions': mask[0].nonzero(as_tuple=False).flatten().tolist(),
                          'levels_completed': raw_observation.get('levels_completed'), 'environment_state': raw_observation.get('state'),
                          'guid': raw_observation.get('guid'), 'episode_id': identity,
                          'transition_action': state.get('pending_action'),
                          'claim': 'observation received; no private game state or rule verification'}
            observed = workspace.observe(latent[0], coordinates, state.get('pending_action'), provenance)
            history = (state.get('history', []) + [latent[0].float().cpu().tolist()])[-int(reason['history_length']):]
            memory, memory_events = workspace.retrieve(latent[0], model.geometry, coordinates,
                                                       count=min(32, int(reason['memory_capacity'])), branch='observed', mode=reason['geometry'])
            if not ledger.consume('queen'):
                raise RuntimeError('Missing Queen budget.')
            decision = model.queen(torch.tensor(history, device=device, dtype=latent.dtype).unsqueeze(0), memory.unsqueeze(0), mask)
            # The fallback is the Queen's legal preference; lower-role calls require reserved budget.
            fallback_id = int(decision['action_logits'][0].argmax())
            fallback_xy = decision['click_logits'][0].argmax(-1).float()*(2.0/63.0)-1.0
            fallback = _action_from_candidate(fallback_id, fallback_xy)
            halt_probability = float(decision['halt_logit'][0].sigmoid())
            trace['queen'] = {'worker_weight': float(decision['worker_weight'][0]),
                              'scout_weight': float(decision['scout_weight'][0]),
                              'halt_probability': halt_probability,
                              'goal_norm': float(decision['goal'][0].float().norm())}
            trace['observed_event_id'] = observed['id']
            trace['memory_events'] = [{'id': e['id'], 'kind': e['kind'], 'branch': e['branch']} for e in memory_events]
            frontier = [{'latent': latent, 'event_id': observed['id'], 'coordinates': coordinates,
                         'branch': f"obs-{observed['id']}", 'score': 0.0, 'first_action': None,
                         'root_event_id': None}]
            best = None
            evaluated = 0
            for depth in range(1, int(reason['max_depth'])+1):
                if depth > 1 and reason['queen_mode']=='learned' and halt_probability > 0.8:
                    break
                expansions = []
                for parent in frontier:
                    roles = [('worker', model.worker, int(reason['worker_candidates']), float(decision['worker_weight'][0]))]
                    if reason['scout_enabled']:
                        roles.append(('scout', model.scout, int(reason['scout_candidates']), float(decision['scout_weight'][0])))
                    if reason['queen_mode']=='learned':
                        roles.sort(key=lambda item: item[3], reverse=True)
                    for role_name, role_model, count, weight in roles:
                        if reason['queen_mode'] == 'learned' and reason['scout_enabled']:
                            count = max(1, math.ceil(count * weight))
                        # Proposal block + refinement energy blocks + final energy/decode + world block.
                        cost = int(role_model.refine_steps) + 4
                        if not ledger.consume(role_name, cost):
                            continue
                        proposals = role_model.propose(parent['latent'], decision['goal'], mask, count)
                        proposed_actions = proposals['actions'][0]
                        proposed_xy = proposals['xy'][0]
                        # Hypotheses predict from the parent belief, not from an invented observed state.
                        predicted = model.world(parent['latent'].expand(count, -1), proposed_actions, proposed_xy)
                        predicted_latents = predicted['next_latent']
                        points = model.geometry.event(predicted_latents, parent['coordinates'])
                        geometry_cost = model.geometry.separation(parent['coordinates'], points,
                                                                 proposals['latents'][0], reason['geometry'])
                        q_logprob = decision['action_logits'][0].float().log_softmax(-1)[proposed_actions]
                        alignment = (proposals['latents'][0].float()-predicted_latents.float()).square().mean(-1)
                        energy = proposals['energies'][0].float()
                        uncertainty = predicted['uncertainty'].float()
                        novelty = proposals.get('novelty', torch.zeros_like(proposals['energies']))[0].float()
                        scores = (parent['score'] + predicted['reward'].float() - energy - 0.2*alignment + 0.1*math.log(max(weight, 1e-6))
                                  - 0.5*uncertainty - 0.01*geometry_cost + 0.2*q_logprob
                                  + (0.05*novelty if role_name=='scout' else 0.0))
                        for index in range(count):
                            action = _action_from_candidate(proposed_actions[index], proposed_xy[index])
                            if not bool(mask[0, action.action]):
                                raise AssertionError('Proposal escaped the hard public action mask.')
                            branch = parent['branch'] + f'/{role_name}-{index}'
                            event = workspace.add(predicted_latents[index], points[index], 'imagined',
                                                  parent['event_id'], branch, action.to_dict(),
                                                  {'source': 'learned_world_model', 'depth': depth,
                                                   'rules_and_success': 'unverified_prediction',
                                                   'role': role_name})
                            item = {'role': role_name, 'depth': depth, 'branch': branch, 'event_id': event['id'],
                                    'parent_id': parent['event_id'], 'action': action.to_dict(),
                                    'energy': float(energy[index]), 'predicted_reward': float(predicted['reward'][index]),
                                    'predicted_done': float(predicted['done_logit'][index].sigmoid()),
                                    'uncertainty': float(uncertainty[index]), 'geometry_cost': float(geometry_cost[index]),
                                    'score': float(scores[index]), 'label': 'imagined'}
                            trace['candidates'].append(item)
                            first_action = action if parent['first_action'] is None else parent['first_action']
                            expanded = {'latent': predicted_latents[index:index+1], 'coordinates': points[index],
                                        'event_id': event['id'], 'branch': branch, 'score': float(scores[index]),
                                        'first_action': first_action,
                                        'root_event_id': event['id'] if parent['root_event_id'] is None else parent['root_event_id']}
                            expansions.append(expanded)
                            evaluated += 1
                if not expansions:
                    break
                expansions.sort(key=lambda item: item['score'], reverse=True)
                frontier = expansions[:2]
                # Choose among equally deep trajectories; earlier depths remain a fallback if budget stops.
                best = frontier[0]
            selected = fallback if best is None else best['first_action']
            if not bool(mask[0, selected.action]):
                raise AssertionError('Selected action is unavailable in the real observation.')
            if best is not None:
                workspace.reinforce(best['root_event_id'], 0.1)
            state = {'episode_id': identity, 'history': history, 'workspace': workspace.state_dict(),
                     'pending_action': selected.to_dict(), 'decision_index': int(state.get('decision_index', 0))+1}
            trace.update({'selected_action': selected.to_dict(), 'selected_root_candidate': None if best is None else best['root_event_id'],
                          'budget': ledger.state_dict(), 'candidate_evaluations': evaluated,
                          'elapsed_seconds': time.monotonic()-started, 'geometry': reason['geometry']})
            return selected, state, trace
    finally:
        model.train(was_training)

## Recovery gate identity and exact continuation

A passed gate is tied to model dimensions, precision, AdamW and all notebook code hashes. Training budgets, stage names and data paths may change without invalidating this architecture check; every individual run still has strict full-config checkpoint matching. The check saves and restores model, optimizer, scheduler, RNG and portable episode/replay extras, then compares the repeated next step.

In [ ]:
def full_model_recovery_signature(cfg):
    return canonical_hash({'model': cfg['model'], 'precision': cfg['train']['precision'], 'optimizer': 'AdamW'})


def require_full_model_gate(cfg, root=ROOT):
    root = Path(root)
    signature = full_model_recovery_signature(cfg)
    path = root / 'reports' / 'full_model_recovery_gates' / f'{signature}.json'
    if not path.exists():
        raise RuntimeError('Run notebook 09 full-model recovery cell before training this architecture.')
    report = json.loads(path.read_text())
    if report.get('status') != 'passed' or report.get('signature') != signature:
        raise RuntimeError('The full-model recovery gate did not pass.')
    if report.get('source_hashes') != source_manifest(root):
        raise RuntimeError('Notebook code changed: rerun 01 and 09 recovery checks.')
    current_runtime = runtime_fingerprint()
    for key in ('python', 'torch', 'numpy', 'cuda_build'):
        if report.get('runtime', {}).get(key) != current_runtime.get(key):
            raise RuntimeError(f'Full-model recovery runtime changed ({key}); rerun notebook 09.')
    device = torch.device(cfg['train']['device'])
    current_device = torch.cuda.get_device_name(device) if device.type == 'cuda' else device.type
    if report.get('device_name') != current_device:
        raise RuntimeError('GPU family changed; rerun notebook 09 full-model recovery.')
    return report


def _recovery_loss(model, grids, actions, xy):
    out = model(grids, actions=actions, xy=xy)
    target = out['latent'].detach().roll(1, dims=-1)
    loss = (F.cross_entropy(out['action_logits'].float(), actions)
            + 0.05*out['click_logits'].float().square().mean()
            + out['value'].float().square().mean()
            + F.mse_loss(out['next_latent'].float(), target.float())
            + out['reward'].float().square().mean()
            + F.binary_cross_entropy_with_logits(out['done_logit'].float(), torch.zeros_like(out['done_logit'].float()))
            + 0.01*out['uncertainty'].float().mean()
            + 0.01*out['worker_energy'].square().mean() + 0.01*out['scout_energy'].square().mean()
            + 0.01*(out['worker_proposal'].float()-target.float()).square().mean()
            + 0.01*(out['scout_proposal'].float()-target.float()).square().mean()
            + 0.01*out['halt_logit'].float().square().mean()
            + out['geometry_regularizer'])
    if not torch.isfinite(loss):
        raise FloatingPointError('Full-model recovery loss is not finite.')
    return loss


def _portable_state_equal(first, second):
    if isinstance(first, torch.Tensor):
        return isinstance(second, torch.Tensor) and torch.equal(first.cpu(), second.cpu())
    if isinstance(first, np.ndarray):
        return isinstance(second, np.ndarray) and np.array_equal(first, second)
    if isinstance(first, dict):
        return isinstance(second, dict) and first.keys()==second.keys() and all(_portable_state_equal(first[k], second[k]) for k in first)
    if isinstance(first, (tuple, list)):
        return isinstance(second, type(first)) and len(first)==len(second) and all(_portable_state_equal(a,b) for a,b in zip(first, second))
    return first == second


def validate_full_model_recovery(cfg, root=ROOT):
    root = Path(root)
    require_recovery_gate(root)
    ensure_layout(root)
    signature = full_model_recovery_signature(cfg)
    report_path = root / 'reports' / 'full_model_recovery_gates' / f'{signature}.json'
    source_hashes = source_manifest(root)
    report = {'status': 'running', 'signature': signature, 'config_hash': canonical_hash(cfg),
              'source_hashes': source_hashes, 'kind': 'synthetic_full_XL_optimizer_continuation_not_capability'}
    atomic_json(report_path, report)
    started = time.monotonic()
    device = torch.device(cfg['train']['device'])
    precision = cfg['train']['precision']
    if precision not in ('bf16', 'fp32'):
        raise ValueError('First draft validates bf16 or fp32; add scaler validation before enabling fp16.')
    if device.type == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('Configured CUDA device unavailable; do not substitute a compact model.')
    previous_deterministic = torch.are_deterministic_algorithms_enabled()
    previous_tf32 = torch.backends.cuda.matmul.allow_tf32
    model = optimizer = scheduler = None
    try:
        seed_everything(int(cfg['train']['seed']))
        torch.use_deterministic_algorithms(True)
        torch.backends.cuda.matmul.allow_tf32 = False
        model = AHRModel(cfg).to(device)
        model.train()
        optimizer = torch.optim.AdamW(model.parameters(), lr=float(cfg['train']['lr']), weight_decay=float(cfg['train']['weight_decay']))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: 1.0)
        run_id = 'full-model-recovery-' + signature[:12] + '-' + str(time.time_ns())
        manager = CheckpointManager(root, run_id, cfg, source_hashes=source_hashes)
        grids = torch.arange(64*64, device=device).reshape(1,64,64).remainder(16).long()
        actions = torch.tensor([6], device=device, dtype=torch.long)
        xy = torch.tensor([[0.25, -0.5]], device=device, dtype=torch.float32)
        fixture_workspace = EventWorkspace(8)
        fixture_workspace.observe(torch.zeros(int(cfg['model']['d_model'])), [0,0,0,0], provenance={'source':'synthetic_recovery_fixture'})
        extra = {'episode_state': {'workspace': fixture_workspace.state_dict(), 'history': [], 'pending_action': None},
                 'replay': {'records': [{'fixture': 'synthetic', 'cursor': 0}], 'position': 1},
                 'sampler': {'position': 1, 'order': [0]}, 'stage': 'full_model_recovery'}

        def one_step():
            if time.monotonic()-started > 720:
                raise TimeoutError('Recovery profile exceeded 12-minute target; latest checkpoint retained.')
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=precision=='bf16'):
                loss = _recovery_loss(model, grids, actions, xy)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg['train']['grad_clip']), error_if_nonfinite=True)
            optimizer.step()
            scheduler.step()
            if device.type == 'cuda':
                torch.cuda.synchronize(device)
            return float(loss.detach())

        with RunLock(root, run_id):
            loss1 = one_step()
            manager.save(model, optimizer, scheduler, step=1, metrics={'loss': loss1}, extra=extra)
            loss2 = one_step()
            expected = {name: value.detach().cpu().clone() for name,value in model.state_dict().items()}
            expected_optimizer = copy.deepcopy(optimizer.state_dict())
            expected_scheduler = copy.deepcopy(scheduler.state_dict())
            expected_rng = capture_rng()
            loaded = manager.load_latest(model, optimizer, scheduler, map_location=device)
            if loaded is None or loaded['step'] != 1 or not _portable_state_equal(loaded['extra'], extra):
                raise AssertionError('Checkpoint failed to restore step or portable episode/replay extras.')
            repeated_loss2 = one_step()
            max_error = max(float((expected[name].float()-value.detach().cpu().float()).abs().max())
                            for name,value in model.state_dict().items())
            optimizer_equal = _portable_state_equal(expected_optimizer, optimizer.state_dict())
            scheduler_equal = _portable_state_equal(expected_scheduler, scheduler.state_dict())
            rng_equal = _portable_state_equal(expected_rng, capture_rng())
            if max_error != 0.0 or repeated_loss2 != loss2 or not optimizer_equal or not scheduler_equal or not rng_equal:
                raise AssertionError(f'Exact continuation mismatch: parameters={max_error}, optimizer={optimizer_equal}, scheduler={scheduler_equal}, RNG={rng_equal}')
            # Exercise full planning only after deterministic optimizer comparison.
            action, planning_state, planning_trace = model.act(
                {'grid': grids[0].cpu().numpy(), 'episode_id': 'synthetic-recovery',
                 'observation_source': 'synthetic_recovery_fixture'},
                available_actions=[1,2,3,4,5,6,7], budget=min(20, int(cfg['reasoning']['max_model_calls'])))
            if planning_trace['budget']['used'] > planning_trace['budget']['maximum']:
                raise AssertionError('Full planner escaped its hard budget.')
            if any(candidate['label'] != 'imagined' for candidate in planning_trace['candidates']):
                raise AssertionError('Predicted candidates lost their epistemic label.')
            if action.action == 6 and (action.x is None or action.y is None):
                raise AssertionError('Click action has no predicted coordinates.')
            portable_planning_state = json.loads(json.dumps(planning_state))
            if portable_planning_state != planning_state:
                raise AssertionError('Full planner episode state is not portable.')
            extra['planner_state'] = portable_planning_state
            manager.save(model, optimizer, scheduler, step=2, metrics={'loss': repeated_loss2}, extra=extra, is_best=True)
            restored_planning = manager.load_latest(model, optimizer, scheduler, map_location=device)
            if restored_planning['extra']['planner_state'] != portable_planning_state:
                raise AssertionError('Actual full-planner event DAG did not round-trip through checkpoint.')
            report.update({'status': 'passed', 'run_id': run_id, 'step_restored': 1, 'step_repeated': 2,
                           'max_parameter_error': max_error, 'loss_before_save': loss1, 'loss_expected': loss2,
                           'loss_repeated': repeated_loss2, 'optimizer_equal': optimizer_equal, 'scheduler_equal': scheduler_equal, 'rng_equal': rng_equal,
                           'episode_replay_equal': True, 'planner_state_equal': True,
                           'planner_budget': planning_trace['budget'], 'planner_candidates': len(planning_trace['candidates']), 'parameter_count': sum(p.numel() for p in model.parameters()),
                           'precision': precision, 'runtime': runtime_fingerprint(),
                           'device_name': torch.cuda.get_device_name(device) if device.type == 'cuda' else device.type,
                           'elapsed_seconds': time.monotonic()-started})
            atomic_json(report_path, report)
            atomic_json(root / 'reports' / 'full_model_recovery_gate.json', report)
            return report
    except BaseException as error:
        report.update({'status': 'failed', 'error': f'{type(error).__name__}: {error}',
                       'elapsed_seconds': time.monotonic()-started})
        atomic_json(report_path, report)
        atomic_json(root / 'reports' / 'full_model_recovery_gate.json', report)
        raise
    finally:
        torch.use_deterministic_algorithms(previous_deterministic)
        torch.backends.cuda.matmul.allow_tf32 = previous_tf32
        # Keep closure bindings defined while releasing the full model references.
        model = optimizer = scheduler = None
        if device.type == 'cuda':
            torch.cuda.empty_cache()

## Run All: bounded validation

**Run All allocates the actual XL model and performs a short synthetic optimizer continuation check on the configured GPU.** First run 00 and 01; keep `CUBLAS_WORKSPACE_CONFIG=:4096:8` set before the kernel initializes CUDA. A failure leaves training blocked and a report to inspect. This cell does not train on game data, launch a background worker or use a remote game API.

In [ ]:
cfg = load_config()
require_recovery_gate(ROOT)
with torch.device('meta'):
    integrated_metadata = AHRModel(cfg)
print({'full_XL_parameters': sum(p.numel() for p in integrated_metadata.parameters()),
       'device': cfg['train']['device'], 'precision': cfg['train']['precision'],
       'all_roles_present': all(hasattr(integrated_metadata, role) for role in ('encoder','world','worker','scout','queen','geometry'))})
del integrated_metadata
full_recovery_report = validate_full_model_recovery(cfg, ROOT)
print(full_recovery_report)
require_full_model_gate(cfg, ROOT)